# 📥 Import ข้อมูลเข้า MySQL - วิธีง่ายที่สุด!

**วิธีใช้:**
1. เปิด Terminal/CMD ไปที่โฟลเดอร์ `desktop-tutorial`
2. รัน: `jupyter notebook simple_import.ipynb`
3. รัน Cell 1-3 ตามลำดับ
4. เสร็จ!

---

**⏱️ ใช้เวลา:** 2-3 นาที  
**📦 Import:** 208,899 rows

---

## Cell 1: ติดตั้ง Libraries และตรวจสอบไฟล์

In [ ]:
import os
import sys
from pathlib import Path

# แสดง working directory
print("="*70)
print("📥 Import ข้อมูลเข้า MySQL Database".center(70))
print("="*70)
print(f"\n📂 Working directory: {os.getcwd()}\n")

# ติดตั้ง libraries
print("📦 ตรวจสอบ libraries...")
try:
    import mysql.connector
    import pandas as pd
    from datetime import datetime
    print("   ✅ Libraries พร้อมใช้งาน\n")
except ImportError:
    print("   ⏳ กำลังติดตั้ง libraries...")
    !pip install mysql-connector-python pandas -q
    import mysql.connector
    import pandas as pd
    from datetime import datetime
    print("   ✅ ติดตั้ง libraries เสร็จแล้ว\n")

# ตรวจสอบไฟล์ CSV
print("="*70)
print("📁 ตรวจสอบไฟล์ CSV...")
print("="*70)

data_files = {
    'ETF List': 'data/etf_list.csv',
    'Benchmarks': 'data/benchmark_portfolios.csv',
    'Holdings': 'data/benchmark_holdings.csv',
    'Price History': 'data/etf_price_history.csv'
}

all_exists = True
missing_files = []

for name, path in data_files.items():
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024 / 1024
        rows = sum(1 for _ in open(path)) - 1
        print(f"✅ {name:15s}: {rows:>8,} rows ({size:5.1f} MB)")
    else:
        print(f"❌ {name:15s}: ไม่พบไฟล์ {path}")
        missing_files.append(path)
        all_exists = False

if not all_exists:
    print("\n" + "="*70)
    print("❌ ERROR: ไม่พบไฟล์ CSV!")
    print("="*70)
    print(f"\n📂 Current directory: {os.getcwd()}")
    print(f"\n❌ ไฟล์ที่ไม่พบ:")
    for f in missing_files:
        print(f"   - {f}")
    print("\n💡 วิธีแก้:")
    print("   1. ปิด Jupyter Notebook")
    print("   2. เปิด Terminal/CMD")
    print("   3. cd ไปที่โฟลเดอร์ desktop-tutorial:")
    print("      cd /path/to/desktop-tutorial")
    print("   4. รัน: jupyter notebook simple_import.ipynb")
    print("="*70)
    raise FileNotFoundError("ไม่พบไฟล์ CSV บางไฟล์")
else:
    print("\n✅ ไฟล์ CSV ครบทั้งหมด!\n")

---

## Cell 2: ตั้งค่า MySQL และเชื่อมต่อ

**แก้ไข password ให้ตรงกับเครื่องคุณ**

In [ ]:
# ========================================
# ตั้งค่า MySQL Connection
# ========================================

MYSQL_CONFIG = {
    'host': '127.0.0.1',
    'port': 3306,
    'user': 'root',
    'password': 'krittanut123456',  # 🔐 แก้ไข password ตรงนี้
    'database': 'portfolio_backtesting'
}

print("="*70)
print("⚙️  MySQL Configuration")
print("="*70)
print(f"Host:     {MYSQL_CONFIG['host']}")
print(f"Port:     {MYSQL_CONFIG['port']}")
print(f"User:     {MYSQL_CONFIG['user']}")
print(f"Database: {MYSQL_CONFIG['database']}")
print("="*70)

# เชื่อมต่อ MySQL
print("\n🔌 กำลังเชื่อมต่อ MySQL...")

try:
    conn = mysql.connector.connect(**MYSQL_CONFIG)
    cursor = conn.cursor(dictionary=True)
    print("✅ เชื่อมต่อ MySQL สำเร็จ\n")

    # ตรวจสอบ tables
    cursor.execute("SHOW TABLES")
    tables = [list(t.values())[0] for t in cursor.fetchall()]
    print(f"✅ พบ {len(tables)} tables ใน database")
    for i, table in enumerate(tables, 1):
        print(f"   {i}. {table}")
    print()

except mysql.connector.Error as e:
    print(f"\n❌ Error: {e}\n")
    print("💡 กรุณาตรวจสอบ:")
    print("   1. MySQL server ทำงานอยู่หรือไม่?")
    print("   2. Password ถูกต้องหรือไม่? (แก้ใน MYSQL_CONFIG ด้านบน)")
    print("   3. Database 'portfolio_backtesting' มีอยู่หรือไม่?")
    raise

print("="*70)
print("🎉 พร้อม Import ข้อมูล!")
print("="*70)
print("\n⚠️  รัน Cell 3 เพื่อเริ่ม Import (ใช้เวลา 2-3 นาที)\n")

---

## Cell 3: Import ข้อมูลทั้งหมด

**⚠️ รันครั้งเดียวเท่านั้น!** ถ้ารันซ้ำจะเกิด duplicate error

ใช้เวลา **2-3 นาที**

In [ ]:
start_time = datetime.now()

print("="*70)
print("🚀 เริ่ม Import ข้อมูล...".center(70))
print("="*70)

try:
    # ========================================
    # 1. ETF Master
    # ========================================
    print("\n📊 [1/4] Import ETF Master...")
    df_etf = pd.read_csv('data/etf_list.csv')
    df_etf = df_etf.sort_values('ticker_symbol').reset_index(drop=True)
    print(f"   📁 อ่านไฟล์: {len(df_etf)} rows")

    for _, row in df_etf.iterrows():
        cursor.execute("""
            INSERT INTO etf_master
            (ticker_symbol, etf_name, asset_class, region, sector, expense_ratio, inception_date)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (
            row['ticker_symbol'],
            row['etf_name'],
            row['asset_class'],
            row['region'],
            row['sector'],
            float(row['expense_ratio']),
            row['inception_date']
        ))

    conn.commit()
    print(f"   ✅ Import สำเร็จ: {len(df_etf)} ETFs")

    # Get mapping
    cursor.execute("SELECT etf_id, ticker_symbol FROM etf_master")
    etf_map = {r['ticker_symbol']: r['etf_id'] for r in cursor.fetchall()}
    print(f"   📋 Mapping: {len(etf_map)} tickers → IDs")

    # ========================================
    # 2. Benchmark Portfolios
    # ========================================
    print("\n📊 [2/4] Import Benchmark Portfolios...")
    df_bench = pd.read_csv('data/benchmark_portfolios.csv')
    df_bench = df_bench.sort_values('benchmark_name').reset_index(drop=True)
    print(f"   📁 อ่านไฟล์: {len(df_bench)} rows")

    for _, row in df_bench.iterrows():
        cursor.execute("""
            INSERT INTO benchmark_portfolios
            (benchmark_name, description, risk_level, target_return, asset_allocation)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            row['benchmark_name'],
            row['description'],
            row['risk_level'],
            float(row['target_return']),
            row['asset_allocation']
        ))

    conn.commit()
    print(f"   ✅ Import สำเร็จ: {len(df_bench)} benchmarks")

    # Get mapping
    cursor.execute("SELECT benchmark_id, benchmark_name FROM benchmark_portfolios")
    bench_map = {r['benchmark_name']: r['benchmark_id'] for r in cursor.fetchall()}
    print(f"   📋 Mapping: {len(bench_map)} benchmarks → IDs")

    # ========================================
    # 3. Benchmark Holdings
    # ========================================
    print("\n📊 [3/4] Import Benchmark Holdings...")
    df_holdings = pd.read_csv('data/benchmark_holdings.csv')
    print(f"   📁 อ่านไฟล์: {len(df_holdings)} rows")

    imported = 0
    for _, row in df_holdings.iterrows():
        benchmark_id = bench_map.get(row['benchmark_name'])
        etf_id = etf_map.get(row['ticker_symbol'])

        if benchmark_id and etf_id:
            cursor.execute("""
                INSERT INTO benchmark_holdings (benchmark_id, etf_id, target_weight)
                VALUES (%s, %s, %s)
            """, (benchmark_id, etf_id, float(row['target_weight'])))
            imported += 1

    conn.commit()
    print(f"   ✅ Import สำเร็จ: {imported} holdings")

    # ========================================
    # 4. Price History
    # ========================================
    print("\n📊 [4/4] Import Price History (ใช้เวลา 1-2 นาที)...")
    df_price = pd.read_csv('data/etf_price_history.csv')
    print(f"   📁 อ่านไฟล์: {len(df_price):,} rows")

    # แปลง ticker → etf_id
    df_price['etf_id'] = df_price['ticker'].map(etf_map)
    df_price = df_price.dropna(subset=['etf_id'])
    print(f"   🔄 แปลง ticker → etf_id: {len(df_price):,} rows")

    # Import เป็น batch
    batch_size = 5000
    total_batches = (len(df_price) + batch_size - 1) // batch_size

    imported_price = 0
    for i in range(0, len(df_price), batch_size):
        batch = df_price.iloc[i:i+batch_size]

        data = [
            (
                int(row['etf_id']),
                row['date'],
                float(row['open']),
                float(row['high']),
                float(row['low']),
                float(row['close']),
                int(row['volume'])
            )
            for _, row in batch.iterrows()
        ]

        cursor.executemany("""
            INSERT INTO price_history
            (etf_id, price_date, open_price, high_price, low_price, close_price, volume)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, data)

        imported_price += len(batch)
        current_batch = i // batch_size + 1
        percent = (imported_price / len(df_price)) * 100
        print(f"   ⏳ Progress: {imported_price:,}/{len(df_price):,} rows ({percent:.1f}%) - Batch {current_batch}/{total_batches}", end='\r')

    conn.commit()
    print(f"\n   ✅ Import สำเร็จ: {imported_price:,} price records")

    # ========================================
    # สรุปผลลัพธ์
    # ========================================
    elapsed = (datetime.now() - start_time).total_seconds()
    print("\n" + "="*70)
    print("🎉 Import เสร็จสมบูรณ์!".center(70))
    print("="*70)
    print(f"\n⏱️  ใช้เวลา: {elapsed:.1f} วินาที ({elapsed/60:.1f} นาที)\n")
    print("📊 สรุปผลลัพธ์:")
    print(f"   ✅ ETF Master:           {len(df_etf):>10,} rows")
    print(f"   ✅ Benchmark Portfolios: {len(df_bench):>10,} rows")
    print(f"   ✅ Benchmark Holdings:   {imported:>10,} rows")
    print(f"   ✅ Price History:        {imported_price:>10,} rows")
    print(f"   {'─'*40}")
    print(f"   📦 Total:                {len(df_etf) + len(df_bench) + imported + imported_price:>10,} rows\n")
    print("="*70)
    print("\n🎉 พร้อมใช้งาน! เปิด main.ipynb หรือ analytics.ipynb ได้เลย\n")

except mysql.connector.IntegrityError as e:
    print(f"\n\n⚠️  IntegrityError: {e}\n")
    print("💡 มีข้อมูลอยู่แล้ว! ถ้าต้องการ import ใหม่ รัน SQL นี้ก่อน:")
    print("""
    DELETE FROM price_history;
    DELETE FROM benchmark_holdings;
    DELETE FROM benchmark_portfolios;
    DELETE FROM etf_master;
    """)
    print("จากนั้นรัน Cell 3 ใหม่")

except Exception as e:
    print(f"\n\n❌ Error: {e}\n")
    import traceback
    traceback.print_exc()

finally:
    # ปิดการเชื่อมต่อ
    if 'cursor' in locals():
        cursor.close()
    if 'conn' in locals():
        conn.close()
    print("\n✅ ปิดการเชื่อมต่อ MySQL แล้ว")

---

## 💡 Tips และ Troubleshooting

### ถ้า Error: ไม่พบไฟล์ CSV

**วิธีแก้:**
1. ปิด Jupyter Notebook
2. เปิด Terminal/CMD
3. cd ไปที่โฟลเดอร์ desktop-tutorial:
   ```bash
   cd /path/to/desktop-tutorial
   ```
4. รัน Jupyter จากโฟลเดอร์นั้น:
   ```bash
   jupyter notebook simple_import.ipynb
   ```

### ถ้า Error: MySQL Connection Failed

1. ตรวจสอบ MySQL server ทำงานหรือไม่
2. แก้ password ใน Cell 2 (`MYSQL_CONFIG`)
3. ตรวจสอบว่าสร้าง database `portfolio_backtesting` แล้วหรือไม่

### ถ้า Error: Duplicate Entry

มีข้อมูลอยู่แล้ว! รัน SQL นี้ใน MySQL Workbench:

```sql
DELETE FROM price_history;
DELETE FROM benchmark_holdings;
DELETE FROM benchmark_portfolios;
DELETE FROM etf_master;
```

จากนั้นรัน Cell 3 ใหม่

---

## ✅ ตรวจสอบว่า Import สำเร็จ

รัน SQL นี้ใน MySQL Workbench:

```sql
SELECT COUNT(*) FROM etf_master;           -- ต้องได้ 50
SELECT COUNT(*) FROM benchmark_portfolios; -- ต้องได้ 35
SELECT COUNT(*) FROM benchmark_holdings;   -- ต้องได้ 114
SELECT COUNT(*) FROM price_history;        -- ต้องได้ 208,700
```